# Topic 4: select, filter, withColumn — Core DataFrame Transformations

> **Phase 3 → Week 3 → Topic 4**

---

## The Spreadsheet Analogy

Think of a DataFrame as a spreadsheet:
- `select` = choose which columns to show (hide others)
- `filter` = turn on AutoFilter to show only matching rows
- `withColumn` = add a new computed column (formula column)
- `withColumnRenamed` = rename a column header
- `drop` = hide a column permanently
- `cast` = change the data type of a cell range
- `when/otherwise` = `=IF(condition, value_if_true, value_if_false)`

---

## The `col()` Function — Your Most-Used Tool

`F.col("column_name")` creates a **Column expression** — the building block of most DataFrame operations.

```python
from pyspark.sql import functions as F

# These are equivalent ways to reference a column:
df["salary"]             # dict-style
df.salary                # attribute-style (breaks for names with spaces)
F.col("salary")          # col() — most explicit, preferred in functions
F.expr("salary * 0.3")   # SQL expression string — for complex expressions
```

**Use `F.col()` in your code.** It's the clearest and most flexible.

---

## `when` / `otherwise` — The CASE WHEN of PySpark

```python
# SQL equivalent:
# CASE WHEN salary > 90000 THEN 'Senior'
#      WHEN salary > 70000 THEN 'Mid'
#      ELSE 'Junior'
# END

F.when(F.col("salary") > 90000, "Senior") \
 .when(F.col("salary") > 70000, "Mid") \
 .otherwise("Junior")
```

Chain as many `.when()` as needed, end with `.otherwise()`. Without `.otherwise()`, non-matching rows get `null`.

---

## Interview Cheat Sheet

**Q: What is the difference between `filter()` and `where()` in Spark?**
> They are identical — `where()` is just an alias for `filter()`. Both accept a Column expression or a SQL string. Use whichever reads more naturally: DataFrame fans use `filter()`, SQL fans use `where()`.

**Q: What is the difference between `select()` and `withColumn()`?**
> `select()` returns a new DataFrame with ONLY the specified columns (drops all others). `withColumn()` adds or replaces ONE column while keeping ALL existing columns. Use `select()` when you want to reshape/reduce the schema; use `withColumn()` when you want to add a derived column.

**Q: What does `F.lit()` do?**
> `F.lit(value)` creates a Column containing a literal constant value — the same value for every row. Example: `df.withColumn("source", F.lit("etl_pipeline"))` adds a column where every row has the string `"etl_pipeline"`. Needed because column expressions expect Column objects, not Python scalars.

**Q: What is `F.expr()` and when would you use it?**
> `F.expr(sql_string)` evaluates a SQL expression string as a Column. Use it for complex expressions that are easier to write in SQL than in the functional API: `F.expr("salary * 0.3 + bonus")` or `F.expr("CASE WHEN salary > 100000 THEN 'high' ELSE 'low' END")`.

**Q: How do you handle null values in a filter?**
> `null` comparisons always return `null` (not True or False) in SQL/Spark. Use `isNull()` or `isNotNull()` explicitly: `df.filter(F.col("name").isNotNull())`. Never use `== None` — it won't work as expected.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date

spark = SparkSession.builder \
    .appName("Week3 - select filter withColumn") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

# Load employees dataset
schema = StructType([
    StructField("id",        IntegerType(), False),
    StructField("name",      StringType(),  True),
    StructField("dept",      StringType(),  True),
    StructField("salary",    IntegerType(), True),
    StructField("country",   StringType(),  True),
    StructField("join_date", DateType(),    True),
    StructField("active",    BooleanType(), True),
])

df = spark.read.schema(schema).option("header", True) \
    .option("dateFormat", "yyyy-MM-dd").csv("data/employees.csv")

print("Dataset loaded:")
df.show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/14 03:19:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Dataset loaded:


+---+-----+-----------+------+---------+----------+------+
| id| name|       dept|salary|  country| join_date|active|
+---+-----+-----------+------+---------+----------+------+
|  1|Alice|Engineering| 95000|    India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|      USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|    India|2022-01-10|  true|
|  4| Dave|  Marketing| 68000|       UK|2019-11-20| false|
|  5|  Eve|      Sales| 55000|    India|2023-02-28|  true|
|  6|Frank|      Sales| 62000|      USA|2021-08-15|  true|
|  7|Grace|Engineering|105000|    India|2018-06-01|  true|
|  8|Henry|  Marketing| 78000|Australia|2022-09-05|  true|
|  9| Isla|Engineering| 91000|       UK|2020-12-01| false|
| 10| Jack|      Sales| 59000|    India|2023-05-20|  true|
+---+-----+-----------+------+---------+----------+------+



## Part 1: `select()` — Choose Columns

In [2]:
# Basic select — choose columns by name
print("=== select() ===")

# Method 1: string names
df.select("name", "dept", "salary").show()

# Method 2: col() expressions — allows aliasing and transformations
df.select(
    F.col("name"),
    F.col("dept"),
    F.col("salary"),
    (F.col("salary") * 0.3).alias("tax"),           # computed column
    F.col("join_date").alias("start_date"),          # rename inline
).show()

# Method 3: expr() — SQL expressions
df.select(
    "name",
    F.expr("salary * 0.3 AS tax"),
    F.expr("UPPER(dept) AS dept_upper"),
).show()

=== select() ===


+-----+-----------+------+
| name|       dept|salary|
+-----+-----------+------+
|Alice|Engineering| 95000|
|  Bob|Engineering| 88000|
|Carol|  Marketing| 72000|
| Dave|  Marketing| 68000|
|  Eve|      Sales| 55000|
|Frank|      Sales| 62000|
|Grace|Engineering|105000|
|Henry|  Marketing| 78000|
| Isla|Engineering| 91000|
| Jack|      Sales| 59000|
+-----+-----------+------+



+-----+-----------+------+-------+----------+
| name|       dept|salary|    tax|start_date|
+-----+-----------+------+-------+----------+
|Alice|Engineering| 95000|28500.0|2021-03-15|
|  Bob|Engineering| 88000|26400.0|2020-07-01|
|Carol|  Marketing| 72000|21600.0|2022-01-10|
| Dave|  Marketing| 68000|20400.0|2019-11-20|
|  Eve|      Sales| 55000|16500.0|2023-02-28|
|Frank|      Sales| 62000|18600.0|2021-08-15|
|Grace|Engineering|105000|31500.0|2018-06-01|
|Henry|  Marketing| 78000|23400.0|2022-09-05|
| Isla|Engineering| 91000|27300.0|2020-12-01|
| Jack|      Sales| 59000|17700.0|2023-05-20|
+-----+-----------+------+-------+----------+



+-----+-------+-----------+
| name|    tax| dept_upper|
+-----+-------+-----------+
|Alice|28500.0|ENGINEERING|
|  Bob|26400.0|ENGINEERING|
|Carol|21600.0|  MARKETING|
| Dave|20400.0|  MARKETING|
|  Eve|16500.0|      SALES|
|Frank|18600.0|      SALES|
|Grace|31500.0|ENGINEERING|
|Henry|23400.0|  MARKETING|
| Isla|27300.0|ENGINEERING|
| Jack|17700.0|      SALES|
+-----+-------+-----------+



In [3]:
# Select ALL columns + extras using * trick
df.select("*", (F.col("salary") * 12).alias("annual_salary")).show(3)

# Select by excluding columns
all_except_id = [c for c in df.columns if c != "id"]
df.select(all_except_id).show(3)

# selectExpr — pure SQL strings, very concise
df.selectExpr(
    "name",
    "salary",
    "salary * 0.3 AS tax",
    "UPPER(dept) AS dept",
    "DATEDIFF(CURRENT_DATE(), join_date) AS days_employed",
).show()

+---+-----+-----------+------+-------+----------+------+-------------+
| id| name|       dept|salary|country| join_date|active|annual_salary|
+---+-----+-----------+------+-------+----------+------+-------------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|      1140000|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|  true|      1056000|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|       864000|
+---+-----+-----------+------+-------+----------+------+-------------+
only showing top 3 rows



+-----+-----------+------+-------+----------+------+
| name|       dept|salary|country| join_date|active|
+-----+-----------+------+-------+----------+------+
|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  Bob|Engineering| 88000|    USA|2020-07-01|  true|
|Carol|  Marketing| 72000|  India|2022-01-10|  true|
+-----+-----------+------+-------+----------+------+
only showing top 3 rows



+-----+------+-------+-----------+-------------+
| name|salary|    tax|       dept|days_employed|
+-----+------+-------+-----------+-------------+
|Alice| 95000|28500.0|ENGINEERING|         1886|
|  Bob| 88000|26400.0|ENGINEERING|         2143|
|Carol| 72000|21600.0|  MARKETING|         1585|
| Dave| 68000|20400.0|  MARKETING|         2367|
|  Eve| 55000|16500.0|      SALES|         1171|
|Frank| 62000|18600.0|      SALES|         1733|
|Grace|105000|31500.0|ENGINEERING|         2904|
|Henry| 78000|23400.0|  MARKETING|         1347|
| Isla| 91000|27300.0|ENGINEERING|         1990|
| Jack| 59000|17700.0|      SALES|         1090|
+-----+------+-------+-----------+-------------+



## Part 2: `filter()` / `where()` — Filter Rows

In [4]:
print("=== filter() / where() ===")

# Single condition
df.filter(F.col("salary") > 80000).show()

# Multiple conditions — AND / OR
df.filter(
    (F.col("salary") > 70000) & (F.col("country") == "India")
).show()

df.filter(
    (F.col("dept") == "Engineering") | (F.col("salary") > 90000)
).show()

# NOT condition
df.filter(~(F.col("dept") == "Sales")).show(3)

# SQL string condition — great for complex expressions
df.filter("salary > 70000 AND country = 'India'").show()
df.where("dept IN ('Engineering', 'Marketing')").show()

=== filter() / where() ===


+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
|  9| Isla|Engineering| 91000|     UK|2020-12-01| false|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
|  9| Isla|Engineering| 91000|     UK|2020-12-01| false|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
+---+-----+-----------+------+-------+----------+------+
only showing top 3 rows



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+---------+----------+------+
| id| name|       dept|salary|  country| join_date|active|
+---+-----+-----------+------+---------+----------+------+
|  1|Alice|Engineering| 95000|    India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|      USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|    India|2022-01-10|  true|
|  4| Dave|  Marketing| 68000|       UK|2019-11-20| false|
|  7|Grace|Engineering|105000|    India|2018-06-01|  true|
|  8|Henry|  Marketing| 78000|Australia|2022-09-05|  true|
|  9| Isla|Engineering| 91000|       UK|2020-12-01| false|
+---+-----+-----------+------+---------+----------+------+



In [5]:
# String filters — startsWith, endsWith, contains, like, rlike
df.filter(F.col("name").startswith("A")).show()
df.filter(F.col("name").endswith("e")).show()
df.filter(F.col("name").contains("a")).show()
df.filter(F.col("dept").like("%ing%")).show()       # SQL LIKE (% = wildcard)
df.filter(F.col("name").rlike("^[AC]")).show()      # regex — names starting with A or C

# isin — IN list
df.filter(F.col("country").isin(["India", "USA"])).show()
df.filter(~F.col("country").isin(["India", "USA"])).show()  # NOT IN

# between
df.filter(F.col("salary").between(60000, 90000)).show()

+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  4| Dave|  Marketing| 68000|     UK|2019-11-20| false|
|  5|  Eve|      Sales| 55000|  India|2023-02-28|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
|  4| Dave|  Marketing| 68000|     UK|2019-11-20| false|
|  6|Frank|      Sales| 62000|    USA|2021-08-15|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
|  9| Isla|Engineering| 91000|     UK|2020-12-01| false|
| 10| Jack|      Sales| 59000|  India|2023-05-20|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+---------+----------+------+
| id| name|       dept|salary|  country| join_date|active|
+---+-----+-----------+------+---------+----------+------+
|  1|Alice|Engineering| 95000|    India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|      USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|    India|2022-01-10|  true|
|  4| Dave|  Marketing| 68000|       UK|2019-11-20| false|
|  7|Grace|Engineering|105000|    India|2018-06-01|  true|
|  8|Henry|  Marketing| 78000|Australia|2022-09-05|  true|
|  9| Isla|Engineering| 91000|       UK|2020-12-01| false|
+---+-----+-----------+------+---------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+-------+----------+------+
| id| name|       dept|salary|country| join_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
|  5|  Eve|      Sales| 55000|  India|2023-02-28|  true|
|  6|Frank|      Sales| 62000|    USA|2021-08-15|  true|
|  7|Grace|Engineering|105000|  India|2018-06-01|  true|
| 10| Jack|      Sales| 59000|  India|2023-05-20|  true|
+---+-----+-----------+------+-------+----------+------+



+---+-----+-----------+------+---------+----------+------+
| id| name|       dept|salary|  country| join_date|active|
+---+-----+-----------+------+---------+----------+------+
|  4| Dave|  Marketing| 68000|       UK|2019-11-20| false|
|  8|Henry|  Marketing| 78000|Australia|2022-09-05|  true|
|  9| Isla|Engineering| 91000|       UK|2020-12-01| false|
+---+-----+-----------+------+---------+----------+------+



+---+-----+-----------+------+---------+----------+------+
| id| name|       dept|salary|  country| join_date|active|
+---+-----+-----------+------+---------+----------+------+
|  2|  Bob|Engineering| 88000|      USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|    India|2022-01-10|  true|
|  4| Dave|  Marketing| 68000|       UK|2019-11-20| false|
|  6|Frank|      Sales| 62000|      USA|2021-08-15|  true|
|  8|Henry|  Marketing| 78000|Australia|2022-09-05|  true|
+---+-----+-----------+------+---------+----------+------+



In [6]:
# NULL handling in filters
null_df = spark.createDataFrame([
    (1, "Alice",  95000),
    (2, None,     88000),
    (3, "Carol",  None),
], ["id", "name", "salary"])

print("=== NULL handling in filter ===")

# WRONG — never use == None
print("df.filter(col('name') == None) — WRONG, returns empty:")
null_df.filter(F.col("name") == None).show()   # returns empty!

# CORRECT — use isNull()
print("df.filter(col('name').isNull()) — CORRECT:")
null_df.filter(F.col("name").isNull()).show()
null_df.filter(F.col("name").isNotNull()).show()

# Filter for actual values (skip nulls automatically)
null_df.filter(F.col("salary") > 90000).show()   # nulls are excluded automatically

=== NULL handling in filter ===
df.filter(col('name') == None) — WRONG, returns empty:
+---+----+------+
| id|name|salary|
+---+----+------+
+---+----+------+

df.filter(col('name').isNull()) — CORRECT:


+---+----+------+
| id|name|salary|
+---+----+------+
|  2|NULL| 88000|
+---+----+------+



+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice| 95000|
|  3|Carol|  NULL|
+---+-----+------+



+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|Alice| 95000|
+---+-----+------+



## Part 3: `withColumn()` — Add or Replace a Column

In [7]:
print("=== withColumn() ===")

# Add a new column
df2 = df.withColumn("tax",          F.col("salary") * 0.3) \
         .withColumn("net_salary",   F.col("salary") - F.col("salary") * 0.3) \
         .withColumn("senior_flag",  F.col("salary") > 90000) \
         .withColumn("source",       F.lit("hr_system"))     # literal constant

df2.select("name", "salary", "tax", "net_salary", "senior_flag", "source").show()

# Replace an existing column (same column name)
df3 = df.withColumn("salary", F.col("salary") * 1.10)  # 10% raise!
print("After 10% raise:")
df3.select("name", "salary").show()

=== withColumn() ===


+-----+------+-------+----------+-----------+---------+
| name|salary|    tax|net_salary|senior_flag|   source|
+-----+------+-------+----------+-----------+---------+
|Alice| 95000|28500.0|   66500.0|       true|hr_system|
|  Bob| 88000|26400.0|   61600.0|      false|hr_system|
|Carol| 72000|21600.0|   50400.0|      false|hr_system|
| Dave| 68000|20400.0|   47600.0|      false|hr_system|
|  Eve| 55000|16500.0|   38500.0|      false|hr_system|
|Frank| 62000|18600.0|   43400.0|      false|hr_system|
|Grace|105000|31500.0|   73500.0|       true|hr_system|
|Henry| 78000|23400.0|   54600.0|      false|hr_system|
| Isla| 91000|27300.0|   63700.0|       true|hr_system|
| Jack| 59000|17700.0|   41300.0|      false|hr_system|
+-----+------+-------+----------+-----------+---------+

After 10% raise:


+-----+------------------+
| name|            salary|
+-----+------------------+
|Alice|104500.00000000001|
|  Bob| 96800.00000000001|
|Carol|           79200.0|
| Dave|           74800.0|
|  Eve| 60500.00000000001|
|Frank|           68200.0|
|Grace|115500.00000000001|
|Henry|           85800.0|
| Isla|100100.00000000001|
| Jack| 64900.00000000001|
+-----+------------------+



In [8]:
# when / otherwise — CASE WHEN
print("=== when() / otherwise() — CASE WHEN ===")

df_graded = df.withColumn(
    "salary_grade",
    F.when(F.col("salary") > 90000, "Senior")
     .when(F.col("salary") > 70000, "Mid")
     .when(F.col("salary") > 55000, "Junior")
     .otherwise("Entry")
)
df_graded.select("name", "salary", "salary_grade").show()

# when based on multiple conditions
df.withColumn(
    "highlight",
    F.when(
        (F.col("salary") > 80000) & (F.col("country") == "India"),
        "High earner in India"
    ).when(
        F.col("active") == False,
        "Inactive"
    ).otherwise("Normal")
).select("name", "salary", "country", "active", "highlight").show()

=== when() / otherwise() — CASE WHEN ===


+-----+------+------------+
| name|salary|salary_grade|
+-----+------+------------+
|Alice| 95000|      Senior|
|  Bob| 88000|         Mid|
|Carol| 72000|         Mid|
| Dave| 68000|      Junior|
|  Eve| 55000|       Entry|
|Frank| 62000|      Junior|
|Grace|105000|      Senior|
|Henry| 78000|         Mid|
| Isla| 91000|      Senior|
| Jack| 59000|      Junior|
+-----+------+------------+



+-----+------+---------+------+--------------------+
| name|salary|  country|active|           highlight|
+-----+------+---------+------+--------------------+
|Alice| 95000|    India|  true|High earner in India|
|  Bob| 88000|      USA|  true|              Normal|
|Carol| 72000|    India|  true|              Normal|
| Dave| 68000|       UK| false|            Inactive|
|  Eve| 55000|    India|  true|              Normal|
|Frank| 62000|      USA|  true|              Normal|
|Grace|105000|    India|  true|High earner in India|
|Henry| 78000|Australia|  true|              Normal|
| Isla| 91000|       UK| false|            Inactive|
| Jack| 59000|    India|  true|              Normal|
+-----+------+---------+------+--------------------+



In [9]:
# Date functions with withColumn — very common in ETL
print("=== Date operations with withColumn ===")

df_dates = df.withColumn("year_joined",     F.year("join_date")) \
              .withColumn("month_joined",    F.month("join_date")) \
              .withColumn("days_employed",   F.datediff(F.current_date(), F.col("join_date"))) \
              .withColumn("years_employed",  F.round(F.datediff(F.current_date(), F.col("join_date")) / 365, 1)) \
              .withColumn("is_senior",       F.col("days_employed") > 365 * 3)

df_dates.select("name", "join_date", "year_joined", "days_employed", "years_employed", "is_senior").show()

=== Date operations with withColumn ===


+-----+----------+-----------+-------------+--------------+---------+
| name| join_date|year_joined|days_employed|years_employed|is_senior|
+-----+----------+-----------+-------------+--------------+---------+
|Alice|2021-03-15|       2021|         1886|           5.2|     true|
|  Bob|2020-07-01|       2020|         2143|           5.9|     true|
|Carol|2022-01-10|       2022|         1585|           4.3|     true|
| Dave|2019-11-20|       2019|         2367|           6.5|     true|
|  Eve|2023-02-28|       2023|         1171|           3.2|     true|
|Frank|2021-08-15|       2021|         1733|           4.7|     true|
|Grace|2018-06-01|       2018|         2904|           8.0|     true|
|Henry|2022-09-05|       2022|         1347|           3.7|     true|
| Isla|2020-12-01|       2020|         1990|           5.5|     true|
| Jack|2023-05-20|       2023|         1090|           3.0|    false|
+-----+----------+-----------+-------------+--------------+---------+



## Part 4: `withColumnRenamed`, `drop`, `alias`, `cast`

In [10]:
# withColumnRenamed — rename a column
df.withColumnRenamed("dept", "department") \
  .withColumnRenamed("join_date", "start_date") \
  .show(3)

# Rename multiple columns at once (Python loop)
rename_map = {"dept": "department", "join_date": "start_date", "active": "is_active"}
df_renamed = df
for old, new in rename_map.items():
    df_renamed = df_renamed.withColumnRenamed(old, new)
df_renamed.show(3)

+---+-----+-----------+------+-------+----------+------+
| id| name| department|salary|country|start_date|active|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|  true|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|  true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|  true|
+---+-----+-----------+------+-------+----------+------+
only showing top 3 rows



+---+-----+-----------+------+-------+----------+---------+
| id| name| department|salary|country|start_date|is_active|
+---+-----+-----------+------+-------+----------+---------+
|  1|Alice|Engineering| 95000|  India|2021-03-15|     true|
|  2|  Bob|Engineering| 88000|    USA|2020-07-01|     true|
|  3|Carol|  Marketing| 72000|  India|2022-01-10|     true|
+---+-----+-----------+------+-------+----------+---------+
only showing top 3 rows



In [11]:
# drop — remove columns
print("drop():")
df.drop("active", "join_date").show(3)

# cast — change data type
print("cast():")
df.withColumn("salary_str",  F.col("salary").cast(StringType())) \
  .withColumn("salary_dbl",  F.col("salary").cast("double")) \
  .withColumn("salary_long", F.col("salary").cast(LongType())) \
  .select("name", "salary", "salary_str", "salary_dbl", "salary_long").show(3)

# alias — rename a column expression
df.select(
    F.col("name").alias("employee_name"),
    (F.col("salary") / 12).alias("monthly_salary"),
).show(3)

drop():


+---+-----+-----------+------+-------+
| id| name|       dept|salary|country|
+---+-----+-----------+------+-------+
|  1|Alice|Engineering| 95000|  India|
|  2|  Bob|Engineering| 88000|    USA|
|  3|Carol|  Marketing| 72000|  India|
+---+-----+-----------+------+-------+
only showing top 3 rows

cast():


+-----+------+----------+----------+-----------+
| name|salary|salary_str|salary_dbl|salary_long|
+-----+------+----------+----------+-----------+
|Alice| 95000|     95000|   95000.0|      95000|
|  Bob| 88000|     88000|   88000.0|      88000|
|Carol| 72000|     72000|   72000.0|      72000|
+-----+------+----------+----------+-----------+
only showing top 3 rows



+-------------+-----------------+
|employee_name|   monthly_salary|
+-------------+-----------------+
|        Alice|7916.666666666667|
|          Bob|7333.333333333333|
|        Carol|           6000.0|
+-------------+-----------------+
only showing top 3 rows



## Part 5: `F.lit()`, `F.col()`, `F.expr()` — The Column Building Blocks

In [12]:
print("=== F.lit() — Literal constant column ===")

df.withColumn("currency",   F.lit("INR")) \
  .withColumn("loaded_at",  F.lit(date(2024, 1, 15)).cast(DateType())) \
  .withColumn("pipeline_v", F.lit(2)) \
  .select("name", "salary", "currency", "loaded_at", "pipeline_v").show(3)

print("\n=== F.expr() — Complex SQL expressions ===")

df.select(
    "name",
    "salary",
    F.expr("salary * 0.3").alias("tax"),
    F.expr("CASE WHEN salary > 90000 THEN 'Senior' ELSE 'Junior' END").alias("grade"),
    F.expr("UPPER(name) || ' - ' || dept").alias("full_label"),
    F.expr("DATEDIFF(CURRENT_DATE, join_date) / 365").alias("years_worked"),
).show()

=== F.lit() — Literal constant column ===


+-----+------+--------+----------+----------+
| name|salary|currency| loaded_at|pipeline_v|
+-----+------+--------+----------+----------+
|Alice| 95000|     INR|2024-01-15|         2|
|  Bob| 88000|     INR|2024-01-15|         2|
|Carol| 72000|     INR|2024-01-15|         2|
+-----+------+--------+----------+----------+
only showing top 3 rows


=== F.expr() — Complex SQL expressions ===


+-----+------+-------+------+-------------------+------------------+
| name|salary|    tax| grade|         full_label|      years_worked|
+-----+------+-------+------+-------------------+------------------+
|Alice| 95000|28500.0|Senior|ALICE - Engineering| 5.167123287671233|
|  Bob| 88000|26400.0|Junior|  BOB - Engineering|5.8712328767123285|
|Carol| 72000|21600.0|Junior|  CAROL - Marketing| 4.342465753424658|
| Dave| 68000|20400.0|Junior|   DAVE - Marketing| 6.484931506849315|
|  Eve| 55000|16500.0|Junior|        EVE - Sales| 3.208219178082192|
|Frank| 62000|18600.0|Junior|      FRANK - Sales| 4.747945205479452|
|Grace|105000|31500.0|Senior|GRACE - Engineering| 7.956164383561644|
|Henry| 78000|23400.0|Junior|  HENRY - Marketing|3.6904109589041094|
| Isla| 91000|27300.0|Senior| ISLA - Engineering|5.4520547945205475|
| Jack| 59000|17700.0|Junior|       JACK - Sales|2.9863013698630136|
+-----+------+-------+------+-------------------+------------------+



## Part 6: Common String Functions

In [13]:
print("=== String Functions ===")

df.select(
    F.col("name"),
    F.upper("name").alias("upper"),
    F.lower("name").alias("lower"),
    F.length("name").alias("len"),
    F.substring("name", 1, 3).alias("first3"),   # 1-indexed!
    F.trim("name").alias("trimmed"),
    F.concat(F.col("name"), F.lit(" @ "), F.col("dept")).alias("label"),
    F.regexp_replace("name", "[aeiou]", "*").alias("masked_vowels"),
    F.split("dept", "i").alias("dept_split"),
).show(truncate=False)

=== String Functions ===


+-----+-----+-----+---+------+-------+-------------------+-------------+---------------+
|name |upper|lower|len|first3|trimmed|label              |masked_vowels|dept_split     |
+-----+-----+-----+---+------+-------+-------------------+-------------+---------------+
|Alice|ALICE|alice|5  |Ali   |Alice  |Alice @ Engineering|Al*c*        |[Eng, neer, ng]|
|Bob  |BOB  |bob  |3  |Bob   |Bob    |Bob @ Engineering  |B*b          |[Eng, neer, ng]|
|Carol|CAROL|carol|5  |Car   |Carol  |Carol @ Marketing  |C*r*l        |[Market, ng]   |
|Dave |DAVE |dave |4  |Dav   |Dave   |Dave @ Marketing   |D*v*         |[Market, ng]   |
|Eve  |EVE  |eve  |3  |Eve   |Eve    |Eve @ Sales        |Ev*          |[Sales]        |
|Frank|FRANK|frank|5  |Fra   |Frank  |Frank @ Sales      |Fr*nk        |[Sales]        |
|Grace|GRACE|grace|5  |Gra   |Grace  |Grace @ Engineering|Gr*c*        |[Eng, neer, ng]|
|Henry|HENRY|henry|5  |Hen   |Henry  |Henry @ Marketing  |H*nry        |[Market, ng]   |
|Isla |ISLA |isla |4 

## Exercises

1. From employees: select only Engineering employees in India earning over 70k. Show name, salary, join_date.
2. Add three columns: `monthly_salary`, `tax_30pct`, `net_monthly`. Chain all with `withColumn`.
3. Add a `seniority` column using `when/otherwise`: Senior if >5 years, Mid if >2 years, Junior otherwise (use `days_employed / 365`).
4. Use `selectExpr` to create a single-shot transformed DataFrame: uppercase name, salary in thousands, department abbreviated to first 3 chars.

In [14]:
# Exercise 1
print("Exercise 1 — Engineering, India, salary > 70k:")
df.filter(
    (F.col("dept") == "Engineering") &
    (F.col("country") == "India") &
    (F.col("salary") > 70000)
).select("name", "salary", "join_date").show()

Exercise 1 — Engineering, India, salary > 70k:


+-----+------+----------+
| name|salary| join_date|
+-----+------+----------+
|Alice| 95000|2021-03-15|
|Grace|105000|2018-06-01|
+-----+------+----------+



In [15]:
# Exercise 3 — seniority with when/otherwise
print("Exercise 3 — Seniority column:")
df.withColumn(
    "days", F.datediff(F.current_date(), F.col("join_date"))
).withColumn(
    "seniority",
    F.when(F.col("days") > 365 * 5, "Senior")
     .when(F.col("days") > 365 * 2, "Mid")
     .otherwise("Junior")
).select("name", "join_date", "days", "seniority").show()

Exercise 3 — Seniority column:


+-----+----------+----+---------+
| name| join_date|days|seniority|
+-----+----------+----+---------+
|Alice|2021-03-15|1886|   Senior|
|  Bob|2020-07-01|2143|   Senior|
|Carol|2022-01-10|1585|      Mid|
| Dave|2019-11-20|2367|   Senior|
|  Eve|2023-02-28|1171|      Mid|
|Frank|2021-08-15|1733|      Mid|
|Grace|2018-06-01|2904|   Senior|
|Henry|2022-09-05|1347|      Mid|
| Isla|2020-12-01|1990|   Senior|
| Jack|2023-05-20|1090|      Mid|
+-----+----------+----+---------+



In [16]:
# Exercise 4 — selectExpr one-shot transform
print("Exercise 4 — selectExpr one-shot:")
df.selectExpr(
    "UPPER(name) AS name",
    "ROUND(salary / 1000, 1) AS salary_k",
    "SUBSTRING(dept, 1, 3) AS dept_abbr",
    "country",
).show()

Exercise 4 — selectExpr one-shot:


+-----+--------+---------+---------+
| name|salary_k|dept_abbr|  country|
+-----+--------+---------+---------+
|ALICE|    95.0|      Eng|    India|
|  BOB|    88.0|      Eng|      USA|
|CAROL|    72.0|      Mar|    India|
| DAVE|    68.0|      Mar|       UK|
|  EVE|    55.0|      Sal|    India|
|FRANK|    62.0|      Sal|      USA|
|GRACE|   105.0|      Eng|    India|
|HENRY|    78.0|      Mar|Australia|
| ISLA|    91.0|      Eng|       UK|
| JACK|    59.0|      Sal|    India|
+-----+--------+---------+---------+

